In [ ]:
%pip install transformers
%pip install torch
%pip install tqdm

In [ ]:
from transformers import AutoModel, AutoTokenizer

model_id = "Qwen/Qwen2-1.5B-Instruct"
model = AutoModel.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

model.save_pretrained("./model")
tokenizer.save_pretrained("./model")

In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from azure.identity import DefaultAzureCredential

subscription_id = "4e80bb93-03b8-4aaa-8394-7c52710f434a"
resource_group = "ces3-posttraining"
workspace_name = "testwest"

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id,
    resource_group,
    workspace_name,
)

model = Model(
    name="hf-qwen-custom-model",
    path="./model",
    type="custom_model",
)

model = ml_client.models.create_or_update(model)

In [ ]:

from azure.ai.ml.entities import ManagedOnlineEndpoint
import datetime
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from azure.identity import DefaultAzureCredential

subscription_id = "4e80bb93-03b8-4aaa-8394-7c52710f434a"
resource_group = "ces3-posttraining"
workspace_name = "testwest"

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id,
    resource_group,
    workspace_name,
)
endpoint_name = f"hf-empty-endpoint-{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    auth_mode="key",  # or "aad_token"
    description="empty inference endpoint",
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment, Model, CodeConfiguration, ProbeSettings
print(f"endpoint: {endpoint_name}")
for e in ml_client.environments.list(name="hf-env"):
  print(e.name, e.version)
  break

print(f"env:{e.name}:{e.version}")
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=model,                 # or "hf-model:1"
    environment=f"{e.name}:{e.version}",  # or "hf-env@latest"
    code_configuration=CodeConfiguration(code="./src", scoring_script="score-empty.py"),
    instance_type="Standard_NC4as_T4_v3",
    instance_count=1,    
    environment_variables={
        "AZUREML_HTTP_PORT": "5001"
    },
    liveness_probe=ProbeSettings(
        initial_delay=300,
        timeout=10,
        period=10,
        failure_threshold=6,
    ),
    readiness_probe=ProbeSettings(
        initial_delay=300,
        timeout=10,
        period=10,
        failure_threshold=6,
    ),


)

ml_client.online_deployments.begin_create_or_update(deployment).result()

In [ ]:

# Fetch endpoint credentials
creds = ml_client.online_endpoints.get_keys(
    name=endpoint_name
)

# API keys (for auth_mode: key)
primary_key = creds.primary_key
secondary_key = creds.secondary_key

print("Primary key:", primary_key)
print("Secondary key:", secondary_key)
# 6) Route traffic to the deployment
endpoint = ml_client.online_endpoints.get(name=endpoint_name)
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()